### Pipeline Generativa: Conditional VAE (cVAE)
Este notebook foca no treinamento de um AutoEncoder Variacional Condicional com o objetivo de gerar amostras sintéticas direcionadas. Utilizaremos a lista de classes críticas identificadas na avaliação da CNN Baseline para realizar um *Data Augmentation* inteligente.

In [7]:
import torch
import torch.optim as optim
import torchvision.transforms as transforms
import torch.utils.data as data
import pandas as pd
import importlib
import optuna
import json
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from tqdm import tqdm
from torchvision.utils import save_image

import config as cfg
importlib.reload(cfg)

import dataset.dataloader as dl
importlib.reload(dl)

import generative.c_vae as cvae
importlib.reload(cvae)

import utils.metrics as mtcs
importlib.reload(mtcs)

import utils.visualization as vis
importlib.reload(vis)

import generative.tuning as tuning
importlib.reload(tuning)

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

cfg.GRAPH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cfg.VAE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.VAE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
cfg.VAE_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
cfg.VAE_AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)

Using device: cpu


### 2. Preparação e Divisão dos Dados
Carregamos o *dataset* aplicando transformações básicas. É fundamental que a semente aleatória (`RANDOM_SEED`) e as proporções de divisão sejam rigorosamente as mesmas utilizadas no classificador *Baseline*. Isto garante que o VAE não tenha acesso às imagens de validação ou teste durante o seu treino, prevenindo o vazamento de dados (*data leakage*). Adicionalmente, a transformação `ToTensor()` converte os valores dos píxeis para o intervalo $[0, 1]$, o que otimiza o cálculo da *Reconstruction Loss* e o funcionamento da camada de ativação *Sigmoid* no final do *decoder*[cite: 42, 83].

In [2]:
# 1. Carregamento e Preparação dos Dados
df = pd.read_csv(cfg.LABELS_PATH)

# O VAE funciona melhor com imagens em [0,1], o que o ToTensor() já garante.
data_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor()
])

# Mantendo exatamente o mesmo split (mesmo RANDOM_SEED) para não vazar dados
train_df, val_df, _ = dl.get_stratified_splits(
    df, test_size=cfg.TEST_SIZE, val_size=cfg.VAL_SIZE, random_state=cfg.RANDOM_SEED
)

train_dataset = dl.ButterflyDataset(df=train_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
val_dataset = dl.ButterflyDataset(df=val_df, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)

train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

n_classes = len(train_dataset.classes)
print(f"Total de classes: {n_classes}")
print(f"Amostras de Treino para o VAE: {len(train_dataset)}")

Total de classes: 75
Amostras de Treino para o VAE: 3639


### 3. Otimização de Hiperparâmetros (Optuna)
Para garantir a melhor performance do nosso cVAE, utilizamos a otimização bayesiana do Optuna. Vamos explorar o tamanho do espaço latente (`latent_dim`) e a taxa de aprendizado (`lr`), utilizando um podador (`MedianPruner`) para interromper precocemente configurações que não minimizem a *Validation Loss*.

In [ ]:
# Configuração do banco de dados do Optuna
OPTUNA_DB_PATH = cfg.VAE_RESULTS_DIR / "optuna_cvae_study.db"
STORAGE_URL = f"sqlite:///{OPTUNA_DB_PATH}" 

# Instanciar a nossa classe Objective SOTA
cvae_objective = tuning.CVAEObjective(
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    n_classes=n_classes,
    img_size=cfg.IMAGE_SIZE,
    epochs_trial=10
)

# Criar e correr o estudo
study = optuna.create_study(
    study_name="cvae_optimization",
    direction='minimize', 
    storage=STORAGE_URL, 
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3)
)

print(f"A iniciar a otimização de hiperparâmetros (Base de Dados: {OPTUNA_DB_PATH})...")
study.optimize(cvae_objective, n_trials=15, show_progress_bar=True)

print("\n--- Melhores Hiperparâmetros ---")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

### 4. Treinamento do Modelo Final (Best cVAE)
Instanciamos o nosso cVAE com os melhores hiperparâmetros descobertos e efetuamos o treino completo. O ficheiro `cvae.py` já cuida de guardar o melhor modelo automaticamente.

In [ ]:
best_latent_dim = study.best_trial.params['latent_dim']
best_lr = study.best_trial.params['lr']

print(f"Instanciando cVAE com latent_dim={best_latent_dim} e lr={best_lr:.6f}")

final_cvae_model = cvae.ConditionalVAE(
    num_classes=n_classes, 
    latent_dim=best_latent_dim, 
    img_channels=3, 
    img_size=cfg.IMAGE_SIZE
).to(device)

optimizer_cvae = torch.optim.Adam(final_cvae_model.parameters(), lr=best_lr)

# Executa o treino e guarda os modelos na nossa diretoria de resultados
trained_cvae_model, cvae_history = cvae.train_cvae(
    model=final_cvae_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_cvae,
    device=device,
    num_epochs=cfg.N_EPOCHS,
    save_dir=cfg.VAE_MODELS_DIR
)

mtcs.evaluate_vae(cvae_history, cfg.VAE_PLOTS_DIR)

### 5. Visualização das Amostras Geradas
Nesta etapa, utilizamos o modelo **cVAE** otimizado para gerar amostras sintéticas exclusivas para as 4 classes onde o classificador apresenta maior dificuldade. O objetivo é inspecionar visualmente a qualidade (nitidez e padrão das asas) antes de injetar estas imagens no dataset da CNN (Active Learning).

In [ ]:
# 1. Carregar as 8 classes críticas do ficheiro alvo
target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
top_4_classes = target_df['Classe'].tolist()[:4]

# 2. Chamar a função externa
vis.generate_cvae_samples_grid(
    model=trained_cvae_model,
    target_classes=top_4_classes,
    class_to_idx=train_dataset.class_to_idx,
    latent_dim=best_latent_dim,
    device=device,
    save_path=cfg.VAE_PLOTS_DIR / 'cvae_grid_1x4.png'
)

### 6. Avaliação Generativa (FID e KID)
Para quantificar a qualidade e a diversidade das imagens geradas, calculamos o *Fréchet Inception Distance* (FID) e o *Kernel Inception Distance* (KID) utilizando uma rede InceptionV3 pré-treinada. Comparamos uma amostra de imagens reais do conjunto de validação com imagens sintéticas geradas pelo cVAE.

In [ ]:
print("A configurar o extrator de features (InceptionV3)...")
# normalize=True é crucial porque o nosso ToTensor() deixa as imagens em [0, 1] e não [0, 255]
fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
kid_metric = KernelInceptionDistance(feature=2048, subset_size=50, normalize=True).to(device)

trained_cvae_model.eval()

print(f"A processar todo o Validation Loader para cálculo preciso de FID/KID...")
with torch.no_grad():
    for real_imgs, labels in tqdm(val_loader, desc="Calculando FID/KID in-memory"):
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)
        batch_size = real_imgs.size(0)
        
        # 1. Atualizar métricas com imagens REAIS
        fid_metric.update(real_imgs, real=True)
        kid_metric.update(real_imgs, real=True)
        
        # 2. Gerar imagens FALSAS condicionadas às mesmas labels
        z = torch.randn(batch_size, best_latent_dim).to(device)
        fake_imgs = trained_cvae_model.decode(z, labels)
        
        # 3. Atualizar métricas com imagens FALSAS
        fid_metric.update(fake_imgs, real=False)
        kid_metric.update(fake_imgs, real=False)

# Calcular os resultados finais
print("\nA extrair e computar a distância das distribuições...")
fid_score = fid_metric.compute()
kid_mean, kid_std = kid_metric.compute()

print("\n" + "="*40)
print(" RESULTADOS DA AVALIAÇÃO GENERATIVA")
print("="*40)
print(f"FID Score: {fid_score.item():.4f}")
print(f"KID Score: {kid_mean.item():.4f} ± {kid_std.item():.4f}")
print("="*40)

# Salvar métricas no disco (para o relatório)
metrics_summary = {
    "FID": fid_score.item(),
    "KID_mean": kid_mean.item(),
    "KID_std": kid_std.item()
}

with open(cfg.VAE_RESULTS_DIR / 'cvae_generative_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=4)

### 7. Geração em Massa (Pool de Candidatos cVAE)

Com o modelo Generativo Condicional (cVAE) treinado e presente em memória, esta etapa foca-se na geração massiva de dados sintéticos para as classes minoritárias/críticas. O modelo amostra o Espaço Latente para gerar um *pool* de 500 candidatas únicas por classe. Estas imagens são temporariamente guardadas no disco e catalogadas num ficheiro CSV, ficando prontas para a fase subsequente de *Hard Sample Mining* (Filtro pelo Oráculo CNN).

In [ ]:
import os
import pandas as pd
import torch
from torchvision.utils import save_image
from tqdm.auto import tqdm

# ==========================================
# 1. PARÂMETROS DA GERAÇÃO DE CANDIDATOS
# ==========================================
CANDIDATES_PER_CLASS = 500

print(f"--- Fase 1: Geração em Massa (Pool de Candidatos) ---")
print(f"Alvo: Gerar {CANDIDATES_PER_CLASS} imagens sintéticas por classe crítica.")

final_cvae_model.eval()

# ==========================================
# 2. CARREGAR CLASSES CRÍTICAS
# ==========================================
target_df = pd.read_csv(cfg.TARGET_CLASSES_PATH)
target_classes = target_df['Classe'].tolist()[:4]
print(f"Classes selecionadas para geração: {target_classes}\n")

candidates_data = []

with torch.no_grad():
    for class_name in target_classes:
        class_idx = train_dataset.class_to_idx[class_name]

        # Prepara a label condicional e amostra o espaço latente
        labels = torch.full((CANDIDATES_PER_CLASS,), class_idx, dtype=torch.long).to(device)
        z = torch.randn(CANDIDATES_PER_CLASS, best_latent_dim).to(device)

        # Decodifica gerando as 500 imagens de uma só vez (GPU-accelerated)
        fake_imgs = final_cvae_model.decode(z, labels)

        for i in tqdm(range(CANDIDATES_PER_CLASS), desc=f"A guardar {class_name}", leave=False):
            safe_class_name = class_name.replace(' ', '_')
            img_filename = f"cvae_candidate_{safe_class_name}_{i:03d}.jpg"
            img_path = cfg.VAE_AUGMENTED_DIR / img_filename

            # Salva a imagem fisicamente no disco
            save_image(fake_imgs[i], img_path)

            # Regista no dicionário para a CNN saber onde as ir buscar
            candidates_data.append({
                'filename': img_filename,
                'label': class_name,
            })

candidates_df = pd.DataFrame(candidates_data)
candidates_csv_path = cfg.VAE_RESULTS_DIR / "cvae_candidates_labels.csv"
candidates_df.to_csv(candidates_csv_path, index=False)

print(f"\n✅ Geração de Candidatos Concluída!")
print(f"Total de {len(candidates_df)} imagens salvas na pasta: {cfg.VAE_AUGMENTED_DIR}")
print(f"Registo salvo em: {candidates_csv_path}")